In [ ]:
"""
    BM25:关键字字面匹配,靠分词词表匹配打分,通用词(动物,的)会带来大量低分无关文档;
    chroma向量检索:语义相似度匹配,理解句子深层含义,即使无相同词汇,同类语义文本也能召回。
"""
# 配置镜像
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
os.environ['HF_HUB_ENABLE_HF_XET'] = '0'

import shutil
import chromadb
from chromadb.utils import embedding_functions
from rank_bm25 import BM25Okapi
import numpy as np
from typing import List
import hanlp
from transformers import BertTokenizer

def patch_hanlp_tokenizer():
    if not hasattr(BertTokenizer, "encode_plus"):#判断当前BertTokenizer有没有encode_plus方法
        def encode_plus(self, text, **kwargs):
            return self(text, **kwargs)
        BertTokenizer.encode_plus = encode_plus
        print("[补丁] 已为 BertTokenizer 添加 encode_plus 方法")

patch_hanlp_tokenizer()

tokenizer=hanlp.load(hanlp.pretrained.tok.COARSE_ELECTRA_SMALL_ZH)#COARSE粗粒度:实体名词不拆分

documents=[
    "猫咪喜欢在阳光下睡觉，它们每天需要很多睡眠。",
    "金毛犬是一种非常友好、活泼的犬种，适合家庭饲养。",
    "大熊猫主要吃竹子，生活在中国的四川、陕西和甘肃。",
    "海豚是聪明的海洋哺乳动物，它们能用声波定位。",
    "企鹅虽然不会飞，但它们是游泳高手，能在水下快速前进。",
    "老虎是大型猫科动物，身上有独特的条纹，是顶级捕食者。",
    "鹦鹉可以模仿人类说话，是一种很受欢迎的宠物鸟。",
]

# 解耦分词器传入BM25类
class BM25Retriever:
    """参数corpus 原始文档列表 tok 分词器"""
    def __init__(self,corpus:List[str], tok):
        self.corpus=corpus
        self.tok = tok
        tokenized_corpus=[self.tok(doc) for doc in corpus] #批量把所有文档分词,生成二维词列表
        self.bm25=BM25Okapi(tokenized_corpus) #构建BM25检索索引:统计全局词频、每篇文档词频、文档平均长度、构建倒排索引、用于后续关键字打分。
    
    """ 参数query进行查找,top_k 得分最高的前top_k条"""
    def retrieve(self, query: str, top_k: int = 5) -> List[str]:
        tokenized_query = self.tok(query) #对用户输入查询语句分词
        scores = self.bm25.get_scores(tokenized_query) #计算每一篇文档和查询词的BM25相关性分数
        top_indicices=np.argsort(scores)[::-1][:top_k] #argsort从小到大排序,返回索引 [::-1]从大到小排序索引,[:topk] 取前几个
        return [self.corpus[i] for i in top_indicices]

# 向量模型。加载多语言句与训练模型 将任意中文文本输入,自动转换为固定维度语义向量。
ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="paraphrase-multilingual-MiniLM-L12-v2")
# 持久化客户端 
client = chromadb.PersistentClient(path="./chroma_db")#磁盘持久化向量库,程序关闭后向量数据不会丢失。

"""复用集合   先尝试获取名为animal_docs的向量集合,集合不存在则新建,并将全部文档写入向量库,自动生成唯一id
              collection.add 内部自动调用上面的ef嵌入函数,批量生成文档向量存入数据库 """
try:
    collection = client.get_collection(name="animal_docs", embedding_function=ef) 
except Exception:
    collection = client.create_collection(name="animal_docs",embedding_function=ef)
    collection.add(documents=documents,ids=[f"id_{i}" for i in range(len(documents))]) #将ids与document的元素一一对应 id_0： id_1：

# 取出集合全部数据
all_data = collection.get() #get的去有效数据,也可以通过下面query查询数据
print("=== ChromaDB 全部存储数据 ===")
print("所有id列表:", all_data["ids"])
print("\n所有文档内容:")
for doc_id, doc_text in zip(all_data["ids"], all_data["documents"]):
    print(f"{doc_id}: {doc_text}")

#向量检索函数 底层逻辑:自动把query转换为向量,计算库内所有文档向量与query向量的余弦相似度,由高到低排序返回
#当有多个query查询语句，二维向量时候记录多个语句,可以进行批量多查询
def chroma_retrieve(query: str, top_k: int = 5) -> List[str]:
    results = collection.query(query_texts=[query],n_results=top_k) #query_texts:输入查询文本列表。n_results:返回相似度最高的top_k条
    return results['documents'][0]   #results['documents']得到的是二维嵌套列表,取[0]拿到本次查询对应的文档一维列表。query计算余弦相似度

if __name__ == "__main__":  #只有当直接运行当前脚本时候才会执行下面测试代码,被其他文件import导入时不会自动执行。
    query = "聪明的海洋动物"

    print(f"查询语句: {query}\n")
    bm25_retriever = BM25Retriever(documents, tokenizer)
    bm25_results = bm25_retriever.retrieve(query, top_k=5)
    print("BM25 检索结果 (top-5):")
    for idx, doc in enumerate(bm25_results, 1):
        print(f"{idx}. {doc}")

    print("\n" + "-"*40 + "\n")
    chroma_results = chroma_retrieve(query, top_k=5)
    print("ChromaDB 向量检索结果 (top-5):")
    for idx, doc in enumerate(chroma_results, 1):
        print(f"{idx}. {doc}")

In [ ]:
"""
    BM25:关键字字面匹配,靠分词词表匹配打分,通用词(动物,的)会带来大量低分无关文档;
    chroma向量检索:语义相似度匹配,理解句子深层含义,即使无相同词汇,同类语义文本也能召回。
"""
# 配置镜像
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
os.environ['HF_HUB_ENABLE_HF_XET'] = '0'

import shutil
import chromadb
from chromadb.utils import embedding_functions
from rank_bm25 import BM25Okapi
import numpy as np
from typing import List
import hanlp
from transformers import BertTokenizer

def patch_hanlp_tokenizer():
    if not hasattr(BertTokenizer, "encode_plus"):#判断当前BertTokenizer有没有encode_plus方法
        def encode_plus(self, text, **kwargs):
            return self(text, **kwargs)
        BertTokenizer.encode_plus = encode_plus
        print("[补丁] 已为 BertTokenizer 添加 encode_plus 方法")
patch_hanlp_tokenizer()

# ===============================================分词模型加载和归一化 ===============================================
tokenizer=hanlp.load(hanlp.pretrained.tok.COARSE_ELECTRA_SMALL_ZH)#COARSE粗粒度:实体名词不拆分

documents=[
    "猫咪喜欢在阳光下睡觉，它们每天需要很多睡眠。",
    "金毛犬是一种非常友好、活泼的犬种，适合家庭饲养。",
    "大熊猫主要吃竹子，生活在中国的四川、陕西和甘肃。",
    "海豚是聪明的海洋哺乳动物，它们能用声波定位。",
    "企鹅虽然不会飞，但它们是游泳高手，能在水下快速前进。",
    "老虎是大型猫科动物，身上有独特的条纹，是顶级捕食者。",
    "鹦鹉可以模仿人类说话，是一种很受欢迎的宠物鸟。",
]
#Min-Max归一化到[0,1]
def min_max_norm(scores):
    if (len(scores)==0): return [0.0]*len(scores)
    min_s,max_s=min(scores),max(scores)
    if max_s-min_s<1e-8:return [0.0]*len(scores)
    return [(s-min_s)/(max_s-min_s) for s in scores]
 #=============================================== BM25 ===============================================

# 解耦分词器传入BM25类
class BM25Retriever:
    """参数corpus 原始文档列表 tok 分词器"""
    def __init__(self,corpus:List[str], tok):
        self.corpus=corpus
        self.tok = tok
        tokenized_corpus=[self.tok(doc) for doc in corpus] #批量把所有文档分词,生成二维词列表
        self.bm25=BM25Okapi(tokenized_corpus) #构建BM25检索索引:统计全局词频、每篇文档词频、文档平均长度、构建倒排索引、用于后续关键字打分。
    
    """ 参数query进行查找,top_k 得分最高的前top_k条"""
    def retrieve(self, query: str, top_k: int = 5):
        tokenized_query = self.tok(query) #对用户输入查询语句分词
        result=[]
        scores = self.bm25.get_scores(tokenized_query) #计算每一篇文档和查询词的BM25相关性分数
        scores=min_max_norm(scores)
        top_indicices=np.argsort(scores)[::-1][:top_k] #argsort从小到大排序,返回索引 [::-1]从大到小排序索引,[:topk] 取前几个
        for i in top_indicices:
            result.append((i,self.corpus[i],scores[i]))
        return result
    
    # 获取全部文档归一BM25分数，用于融合
    def get_all_norm_bm25(self, query:str):
        token_q = self.tok(query)
        raw_scores = self.bm25.get_scores(token_q)
        return min_max_norm(raw_scores)


# =============================================== chroma_db ===============================================

ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="paraphrase-multilingual-MiniLM-L12-v2")
client = chromadb.PersistentClient(path="./chroma_db")#磁盘持久化向量库,程序关闭后向量数据不会丢失。

"""复用集合   先尝试获取名为animal_docs的向量集合,集合不存在则新建,并将全部文档写入向量库,自动生成唯一id
              collection.add 内部自动调用上面的ef嵌入函数,批量生成文档向量存入数据库 """
try:
    collection = client.get_collection(name="animal_docs", embedding_function=ef) 
except Exception:
    collection = client.create_collection(name="animal_docs",embedding_function=ef)
    collection.add(documents=documents,ids=[str(i) for i in range(len(documents))]) #将ids与document的元素一一对应 id_0： id_1：

"""向量检索函数 底层逻辑:自动把query转换为向量,计算库内所有文档向量与query向量的余弦相似度,由高到低排序返回
    当有多个query查询语句,二维向量时候记录多个语句,可以进行批量多查询"""

def chroma_retrieve_with_score(query: str, top_k: int = 5):
    results=collection.query(query_texts=[query],n_results=top_k,include=["documents","distances"])
    #查询第一个query
    docs,dists,ids=results["documents"][0],results["distances"][0],results["ids"][0];res=[]
    raw_sims=[1-d for d in dists]  #1-d 距离相似度
    sim_scores=min_max_norm(raw_sims)
    for doc,sim_score,doc_id in zip(docs,sim_scores,ids):
        res.append((doc_id,doc,sim_score))
    return res

# ===============================================融合检索 ===============================================
# BM25 0.5 + 向量0.5 加权融合检索，传入全局bm25实例，避免重复构建

def hybrid_retrieve(bm25_retriever, query:str, top_k=5, w_bm25=0.5, w_vec=0.5):
    all_bm25_norm = bm25_retriever.get_all_norm_bm25(query)  # 获取全部文档归一BM25分数
    bm25_map = {str(idx): all_bm25_norm[idx] for idx in range(len(all_bm25_norm))} #序号:分数 0:0.0 1:0.0923等 构建id->bm25分映射
    vec_list = chroma_retrieve_with_score(query, top_k)     # 获取向量检索结果（带归一相似度）
    hybrid_result = []
    for doc_id, doc_text, vec_norm_score in vec_list:
        bm25_norm_score = bm25_map.get(doc_id, 0.0)  #要去字典里面区配doc_id键对应的值，若找不到返回0.0
        score = w_bm25 * bm25_norm_score + w_vec * vec_norm_score  #加权总分
        hybrid_result.append((doc_id,doc_text,score))
    hybrid_result.sort(reverse=True, key=lambda x: x[2]) # 按融合总分降序,X[2] 遍历每条数据、取每条内部的第 2 下标字段作为排序依据；
    """  上面lamda等价于
    def get_score(x):
    return x[2]
    hybrid_result.sort(reverse=True, key=get_score)"""
    return hybrid_result
# ===============================================测试 ===============================================
if __name__ == "__main__":  #只有当直接运行当前脚本时候才会执行下面测试代码,被其他文件import导入时不会自动执行。
    query = "聪明的海洋动物"
    bm25_retriever = BM25Retriever(documents, tokenizer)  #全局只初始化一次bm25

    print(f"查询语句: {query}\n")
    bm25_results = bm25_retriever.retrieve(query, top_k=5)
    print("BM25 检索结果 (top-5):")
    for id,text,score in bm25_results:
        print(f"id:{id:<6} 文本:{text:<30} 分数:{score:>8.4f}")

    print("\n" + "-"*40 + "\n")
    chroma_results = chroma_retrieve_with_score(query, top_k=5)
    print("ChromaDB 向量检索结果 (top-5):")

    for id,text,score in chroma_results:
        print(f"id:{id:<6} 文本:{text:<30} 分数:{score:>8.4f}")

    # 新增打印0.5:0.5加权融合结果，传入全局bm25实例
    print("\n" + "="*40 + "\n")
    hybrid_res = hybrid_retrieve(bm25_retriever, query, top_k=5, w_bm25=0.5, w_vec=0.5)
    print("BM25(0.5) + 向量(0.5) 加权融合结果 (top-5):")
    for id,text,score in hybrid_res:
        print(f"id:{id:<6} 文本:{text:<30} 分数:{score:>8.4f}")

查询语句: 聪明的海洋动物

BM25 检索结果 (top-5):
id:3      文本:海豚是聪明的海洋哺乳动物，它们能用声波定位。         分数:  1.0000
id:5      文本:老虎是大型猫科动物，身上有独特的条纹，是顶级捕食者。     分数:  0.4723
id:1      文本:金毛犬是一种非常友好、活泼的犬种，适合家庭饲养。       分数:  0.0923
id:2      文本:大熊猫主要吃竹子，生活在中国的四川、陕西和甘肃。       分数:  0.0896
id:6      文本:鹦鹉可以模仿人类说话，是一种很受欢迎的宠物鸟。        分数:  0.0896

----------------------------------------

ChromaDB 向量检索结果 (top-5):
id:3      文本:海豚是聪明的海洋哺乳动物，它们能用声波定位。         分数:  1.0000
id:4      文本:企鹅虽然不会飞，但它们是游泳高手，能在水下快速前进。     分数:  0.4215
id:6      文本:鹦鹉可以模仿人类说话，是一种很受欢迎的宠物鸟。        分数:  0.2275
id:1      文本:金毛犬是一种非常友好、活泼的犬种，适合家庭饲养。       分数:  0.1746
id:5      文本:老虎是大型猫科动物，身上有独特的条纹，是顶级捕食者。     分数:  0.0000


{'0': np.float64(0.0), '1': np.float64(0.09233087879340153), '2': np.float64(0.08956095242959948), '3': np.float64(1.0), '4': np.float64(0.0), '5': np.float64(0.47227566625211), '6': np.float64(0.08956095242959948)}
BM25(0.5) + 向量(0.5) 加权融合结果 (top-5):
id:3      文本:海豚是聪明的海洋哺乳动物，它们能用声波定位。         分数:  1.0000
id:5      文本:老虎是大型猫科动物，身上有独特

In [22]:
"""
    BM25:关键字字面匹配,靠分词词表匹配打分,通用词会带来低分无关文档;
    chroma向量检索:语义匹配,无相同词汇也能召回同类
"""
# 配置镜像
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
os.environ['HF_HUB_ENABLE_HF_XET'] = '0'

import chromadb
import numpy as np
import hanlp
from rank_bm25 import BM25Okapi
from chromadb.utils import embedding_functions
from transformers import BertTokenizer
from operator import itemgetter

def patch_hanlp_tokenizer():
    if not hasattr(BertTokenizer, "encode_plus"):
        BertTokenizer.encode_plus = lambda self, text, **kwargs: self(text,**kwargs)
patch_hanlp_tokenizer()

# 分词与文档
tokenizer = hanlp.load(hanlp.pretrained.tok.COARSE_ELECTRA_SMALL_ZH)
documents = [
    "猫咪喜欢在阳光下睡觉，它们每天需要很多睡眠。",          # 0
    "金毛犬是一种非常友好、活泼的犬种，适合家庭饲养。",      # 1
    "大熊猫主要吃竹子，生活在中国四川、陕西和甘肃。",        # 2
    "海豚是聪明的海洋哺乳动物，它们能用声波定位。",          # 3
    "企鹅虽然不会飞，但它们是游泳高手，能在水下快速前进。",  # 4
    "老虎是大型猫科动物，身上有独特的条纹，是顶级捕食者。",  # 5
    "鹦鹉可以模仿人类说话，是一种很受欢迎的宠物鸟。"         # 6
]

# MinMax归一
def min_max_norm(scores):
    if len(scores) == 0:
        return [0.0] * len(scores)
    s_min, s_max = min(scores), max(scores)
    if s_max - s_min < 1e-8:
        return [0.0] * len(scores)
    return [(s - s_min) / (s_max - s_min) for s in scores]

# ========================================BM25检索类========================================
class BM25Retriever:
    def __init__(self, corpus, tok):
        self.corpus = corpus
        self.tok = tok
        tokenized = [self.tok(doc) for doc in corpus]
        self.bm25 = BM25Okapi(tokenized)              #构建BM25检索索引

    def retrieve(self, query, top_k=3):
        tokens = self.tok(query)
        scores = min_max_norm(self.bm25.get_scores(tokens)) #self.bm25.get_scores(tokens) 计算分数
        idx = np.argsort(scores)[::-1][:top_k]
        return [(i, self.corpus[i], scores[i]) for i in idx]

    def get_all_norm_score(self, query):
        tokens = self.tok(query)
        return min_max_norm(self.bm25.get_scores(tokens))

# ========================================Chroma初始化========================================
ef = embedding_functions.SentenceTransformerEmbeddingFunction("paraphrase-multilingual-MiniLM-L12-v2")
client = chromadb.PersistentClient("./chroma_db")
try:
    collections = client.get_collection("animal_docs", embedding_function=ef)
except Exception:
    collections = client.create_collection("animal_docs", embedding_function=ef)
    collections.add(documents=documents, ids=[str(i) for i in range(len(documents))])

def vec_retrieve(query, top_k=3):
    res = collections.query(query_texts=[query], n_results=top_k, include=["documents", "distances"])
    ids = res["ids"][0]
    texts = res["documents"][0]
    dists = res["distances"][0]
    sims = min_max_norm([1 - d for d in dists])
    return [(int(i), txt, sim) for i, txt, sim in zip(ids, texts, sims)]

# 加权融合检索
def hybrid_retrieve(bm25_obj, query, top_k=3, w_bm25=0.5, w_vec=0.5):
    all_bm25_sc = bm25_obj.get_all_norm_score(query)
    bm25_map = {i: all_bm25_sc[i] for i in range(len(all_bm25_sc))}
    vec_res = vec_retrieve(query, top_k=7)
    hybrid = []
    for doc_id, text, vec_sc in vec_res:
        bm_sc = bm25_map.get(doc_id, 0.0)
        total = w_bm25 * bm_sc + w_vec * vec_sc
        hybrid.append((doc_id, text, total))
    hybrid.sort(reverse=True, key=itemgetter(2))
    return hybrid[:top_k]

# ========================================评测部分========================================
if __name__ == "__main__":
    bm25_ret = BM25Retriever(documents, tokenizer)
    """
    test_queries = [
        {"q": "海里聪明的海洋哺乳动物", "target": 3},
        {"q": "适合家养温顺宠物狗", "target": 1},
        {"q": "吃竹子的国宝熊猫", "target": 2},
        {"q":"爱吃素的黑白动物","target":2},
        {"q": "不会飞擅长游泳的海鸟", "target": 4},
        {"q": "会学说话的宠物小鸟", "target": 6},
        {"q": "带条纹大型野生猫科猛兽", "target": 5},
        {"q": "白天爱睡觉的家养小猫", "target": 0},
        {"q": "水下靠声波交流的动物", "target": 3},
        {"q": "四川特产珍稀保护动物", "target": 2},
        {"q": "家庭陪伴型中型伴侣犬", "target": 1},
    ]#10条测试query，附带正确目标文档id
   
    test_queries = [
        {"q": "海洋里会用工具捕食的动物", "target": 3},
        {"q": "中型短毛家庭伴侣犬", "target": 1},
        {"q": "亚洲黑白素食野生动物", "target": 2},
        {"q": "潜水捕鱼的水禽类", "target": 4},
        {"q": "能模仿人声的小型笼养鸟", "target": 6},
        {"q": "黄黑条纹的顶级掠食者", "target": 5},
        {"q": "昼伏夜出的毛茸茸家养宠物", "target": 0},
        {"q": "用回声定位的水生哺乳类", "target": 3},
        {"q": "中国西南部的特有濒危物种", "target": 2},
        {"q": "家庭常见的忠诚中型犬种", "target": 1},
    ] """
    test_queries = [
    # 词汇不匹配但语义相关（向量应该比 BM25 强）
    {"q": "海洋中的智多星", "target": 3},           # BM25 可能匹配不到"海豚"
    {"q": "家里最忠诚的四脚朋友", "target": 1},     # BM25 可能匹配不到"狗"
    {"q": "竹林里的黑白胖子", "target": 2},         # BM25 可能匹配不到"熊猫"
    
    # 词汇匹配但语义不同（BM25 应该比向量强）
    {"q": "潜水艇", "target": 4},                   # BM25 命中"潜水"，向量可能偏
    {"q": "老虎机", "target": 5},                   # BM25 命中"老虎"，向量可能偏
    {"q": "猫眼", "target": 0},                     # BM25 命中"猫"，向量可能偏
    
    # 中性难度
    {"q": "南极不会飞的鸟", "target": 4},           # 企鹅
    {"q": "森林之王", "target": 5},                 # 老虎
    {"q": "宠物中的话痨", "target": 6},             # 鹦鹉
    {"q": "国宝的日常", "target": 2},               # 熊猫
    ]  
    total_vec_hit = 0
    total_bm25_hit = 0
    total_hybrid_hit = 0

    for idx, item in enumerate(test_queries, 1):  #idx从1开始,返回(索引,数据)
        q = item["q"]
        target_id = item["target"]
        print(f"\n测试{idx}--Query:{q}--目标文档id={target_id}")

        # 1.纯向量top3
        vec_top3 = vec_retrieve(q, top_k=2)
        vec_ids = [x[0] for x in vec_top3]
        vec_hit = 1 if target_id in vec_ids else 0
        total_vec_hit += vec_hit
        print(f"纯向量top3文档id:{vec_ids} 命中:{vec_hit}")

        # 2.纯BM25 top3
        bm25_top3 = bm25_ret.retrieve(q, top_k=2)
        bm25_ids = [x[0] for x in bm25_top3]
        bm_hit = 1 if target_id in bm25_ids else 0
        total_bm25_hit += bm_hit
        print(f"纯BM25 top3文档id:{bm25_ids} 命中:{bm_hit}")

        # 3.加权融合top3
        hybrid_top3 = hybrid_retrieve(bm25_ret, q,top_k=2,w_bm25=0.5,w_vec=0.5)
        hybrid_ids = [x[0] for x in hybrid_top3]# x=(3, '海豚是聪明的海洋哺乳动物，它们能用声波定位。', np.float64(1.0)) x[0]=3
        hybrid_hit = 1 if target_id in hybrid_ids else 0
        total_hybrid_hit += hybrid_hit
        print(f"加权融合top3文档id:{hybrid_ids} 命中:{hybrid_hit}")
    
    alphas=[0.1,0.3,0.5,0.7,0.9] #alpha
    res=[]

    for alpha in alphas:  #不同参数加权融合的结果
        total_hybrid_hit=0
        for idx, item in enumerate(test_queries, 1):
            q = item["q"]
            target_id = item["target"]
            hybrid_top3 = hybrid_retrieve(bm25_ret, q,top_k=2,w_vec=alpha,w_bm25=1-alpha)
            hybrid_ids = [x[0] for x in hybrid_top3]
            hybrid_hit = 1 if target_id in hybrid_ids else 0
            total_hybrid_hit += hybrid_hit
        res.append(total_hybrid_hit)
    print(res)
    # 汇总总命中数（10条query总命中）
    print("\n==================== 汇总统计 ====================")
    print(f"总测试query数量:10")
    print(f"纯向量检索top3总命中数:{total_vec_hit}")
    print(f"纯BM25检索top3总命中数:{total_bm25_hit}")
    for i in range(np.size(alphas)):
        print(f"双路加权融合top3总命中数{alphas[i]}*vec+{1-alphas[i]:.1f}*bm25:{res[i]}")


测试1--Query:海洋中的智多星--目标文档id=3
纯向量top3文档id:[3, 4] 命中:1
纯BM25 top3文档id:[np.int64(3), np.int64(1)] 命中:1
加权融合top3文档id:[3, 4] 命中:1

测试2--Query:家里最忠诚的四脚朋友--目标文档id=1
纯向量top3文档id:[1, 6] 命中:1
纯BM25 top3文档id:[np.int64(3), np.int64(1)] 命中:1
加权融合top3文档id:[1, 6] 命中:1

测试3--Query:竹林里的黑白胖子--目标文档id=2
纯向量top3文档id:[2, 5] 命中:1
纯BM25 top3文档id:[np.int64(3), np.int64(1)] 命中:0
加权融合top3文档id:[5, 6] 命中:0

测试4--Query:潜水艇--目标文档id=4
纯向量top3文档id:[4, 3] 命中:1
纯BM25 top3文档id:[np.int64(6), np.int64(5)] 命中:0
加权融合top3文档id:[4, 3] 命中:1

测试5--Query:老虎机--目标文档id=5
纯向量top3文档id:[6, 5] 命中:1
纯BM25 top3文档id:[np.int64(6), np.int64(5)] 命中:1
加权融合top3文档id:[6, 5] 命中:1

测试6--Query:猫眼--目标文档id=0
纯向量top3文档id:[5, 0] 命中:1
纯BM25 top3文档id:[np.int64(6), np.int64(5)] 命中:0
加权融合top3文档id:[5, 0] 命中:1

测试7--Query:南极不会飞的鸟--目标文档id=4
纯向量top3文档id:[4, 6] 命中:1
纯BM25 top3文档id:[np.int64(4), np.int64(6)] 命中:1
加权融合top3文档id:[4, 6] 命中:1

测试8--Query:森林之王--目标文档id=5
纯向量top3文档id:[5, 6] 命中:1
纯BM25 top3文档id:[np.int64(6), np.int64(5)] 命中:1
加权融合top3文档id:[5, 6] 命中:1

测试9